In [2]:
!pip install pennylane datasets


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 934.3/934.3 kB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 105.2 MB/s eta 0:00:00


In [3]:
from datasets import load_dataset
import pennylane as qml
from pennylane import numpy as np

# Charger le dataset
ds = load_dataset("Genius-Society/Pima")

# Extraire X et y
X = np.array([[s['Pregnancies'], s['Glucose'], s['BloodPressure'],
               s['SkinThickness'], s['Insulin'], s['BMI'],
               s['DiabetesPedigreeFunction'], s['Age']]
              for s in ds['train']], dtype=float)

y = np.array([s['Outcome'] for s in ds['train']])


/usr/local/lib/python3.12/dist-packages/pennylane/__init__.py:209: RuntimeWarning: PennyLane is not yet compatible with JAX versions > 0.6.2. You have version 0.7.2 installed. Please downgrade JAX to 0.6.2 to avoid runtime errors using python -m pip install jax~=0.6.0 jaxlib~=0.6.0
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

validation.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/614 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/77 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/77 [00:00<?, ? examples/s]

In [4]:
def normalize(x):
    return x / np.max(x)



In [7]:
n_qubits = 8
dev = qml.device("default.qubit", wires=n_qubits)
@qml.qnode(dev)
def directional_entangled_encoding(x):
    x = normalize(x)

    for i in range(n_qubits):
        qml.RX(x[i], wires=i)
        qml.RY(x[i], wires=i)
        qml.RZ(x[i], wires=i)

    # Entanglement en chaîne
    for i in range(n_qubits - 1):
        qml.CNOT(wires=[i, i + 1])

    return qml.state()





In [8]:
state = directional_entangled_encoding(X[0])
print(state)



[ 4.01696203e-01-4.73245819e-01j  4.31987301e-03-6.67261546e-02j
 -8.71264661e-05-9.97503267e-05j -9.67807452e-05-1.22570701e-03j
 -1.30507349e-04-1.80639004e-04j -2.28902464e-05-7.23132904e-06j
 -5.62021031e-03-1.07375167e-02j  1.75856772e-02-1.11126888e-01j
  3.39859832e-03-7.53237087e-02j -4.54618575e-03-6.73052827e-03j
 -1.57853789e-05-3.10355759e-06j -1.00428254e-04-1.10538891e-04j
 -1.56145674e-04-8.09055219e-04j -6.54864405e-05-5.99139337e-05j
 -2.11465327e-03-4.47613749e-02j  2.32119030e-01-3.45221549e-01j
 -1.55424093e-03-4.99807889e-02j -3.34594682e-03-4.22124350e-03j
 -1.05950255e-05-1.25559058e-06j -7.19890861e-05-6.80273098e-05j
 -1.75567942e-05-3.74676845e-06j -1.75863406e-06+8.04203730e-07j
 -9.15866618e-04-3.38163056e-04j -4.95135878e-03-7.59140335e-03j
 -6.07762740e-03-1.20815429e-02j -1.30175789e-03-6.53990509e-04j
 -2.74172212e-06+8.99582621e-07j -2.54098362e-05-8.48015876e-06j
 -9.54174617e-05-1.12864258e-04j -1.55015876e-05-3.62657266e-06j
 -4.25840130e-03-6.816735

In [9]:
# Extraction des features quantiques pour tout le dataset
# Note : cela peut prendre un peu de temps selon la puissance de calcul
X_quantum = []
for row in X:
    state = directional_entangled_encoding(row)
    # On prend la magnitude (probabilités) pour avoir des nombres réels
    X_quantum.append(np.abs(state)**2)

X_quantum = np.array(X_quantum)
print(f"Nouvelle dimension des données : {X_quantum.shape}") # (768, 256)

Nouvelle dimension des données : (614, 256)


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
X_train_q, X_test_q, y_train, y_test = train_test_split(X_quantum, y, test_size=0.2, random_state=42)

clf_quantum = DecisionTreeClassifier(max_depth=5, random_state=42)
clf_quantum.fit(X_train_q, y_train)

y_pred_q = clf_quantum.predict(X_test_q)
print(f"Précision avec Directional Entangled Encoding : {accuracy_score(y_test, y_pred_q):.2%}")

Précision avec Directional Entangled Encoding : 63.41%
